> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Synthetic — genomic and matrix residual experts

The public expert baseline is the genetic mean: training trait mean plus mvBLUP breeding value. Rejected experts return that baseline, including for partial rows. It differs from mvPredict’s full conditional phenotype posterior. Callers providing `blup=` must supply this phenotype-scale genetic mean; omitting it lets the package compute it.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
import vigor
print("VIGOR", vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__
SEED=42
cfg=vigor.simulation_scenario("complex")
cfg.update(dict(heritability=.55, compute_grms=True))
sim=vigor.simulate_genomic_dataset(n_samples=320,n_variants=1000,n_traits=8,seed=SEED,genotype_kwargs={"n_chromosomes":5,"generations":100},trait_kwargs=cfg)
X=sim.genotypes.to_pandas(); Y=sim.phenotypes.to_pandas()
Ka=sim.grms.sel(kernel="additive").to_pandas().to_numpy(float); Kd=sim.grms.sel(kernel="dominance").to_pandas().to_numpy(float)
obs=[c for c in sim.environment.to_pandas() if str(c).startswith("observed")][0]
e=sim.environment.to_pandas()[obs].to_numpy(float); Kg=Ka*np.outer(e,e); Kg/=np.mean(np.diag(Kg))
kernels={"additive":Ka,"dominance":Kd,"gxe":Kg}
rng=np.random.default_rng(20260918); perm=rng.permutation(len(Y)); fit_idx=perm[:224]; unseen_idx=perm[224:272]; partial_idx=perm[272:]
fitmask=np.zeros(len(Y),bool); fitmask[fit_idx]=True
train=Y.copy(); train.iloc[~fitmask,:]=np.nan
pred_traits=train.copy(); held=np.zeros(Y.shape,bool)
for i in partial_idx:
    reveal=rng.choice(Y.shape[1],Y.shape[1]//2,replace=False); pred_traits.iloc[i,reveal]=Y.iloc[i,reveal]; held[i,:]=True; held[i,reveal]=False
fit=vigor.genetic_varcov_PSD(kernels,train,rank_G={k:Y.shape[1] for k in kernels},rank_E=Y.shape[1],solver="lbfgs",variance_closure="auto",variance_closure_strength=1.0,analytic_se="final",analytic_se_probes=16,analytic_trace_probes=8,n_boot=0,random_state=SEED,progress=False,blas_threads=2)

base=vigor.mvBLUP(train,kernels,fit,E=None,missing="exact",accuracy="exact",kernel_rank="full",blas_threads=2)["total"].to_pandas()
# Public expert baseline is mean + breeding value on phenotype units.
base=base.add(train.mean(axis=0),axis=1)
features=vigor.make_genomic_features(kernels,fit_mask=fitmask,n_grm_components=8,genotypes=X,selected_snps=list(X.columns),include_dominance=True,index=Y.index,random_state=SEED)["features"].to_pandas()
expert=vigor.fit_blup_prediction_experts(train,kernels,fit,E=None,blup=base,genomic_features=features,feature_methods=(),matrix_methods=("iterative_ridge",),n_splits=2,selection_cv=2,random_state=SEED,crossfit_refit_covariance=True,fit_phenotype_per_trait=False,genomic_strategy="screened_ridge",genomic_screen_top_k=64,genomic_screen_min_gain=0.0,genomic_screen_min_z=0.0,min_matrix_gain=0.0)
out=vigor.predict_blup_prediction_experts(expert,pred_traits,GRM=kernels,genomic_features=features)
print("modes",expert.attrs["unseen_mode"],expert.attrs["partial_mode"],"features",features.shape)
